# Week 20 workbook --- GRPO, RLOO, RLVR

This workbook has two explicit exercise modes, controlled by `REFERENCE_MODE`
in the setup cell.

- **Reference (`True`, default):** supplied implementations demonstrate the
  calculations. Passing these checks verifies the reference, not your code.
- **Learner (`False`):** complete each scaffold's function, then restart and
  run all cells. The corresponding checks call your function. An unfinished
  scaffold raises `NotImplementedError`; there is no reference fallback.
  In a reference-wrapper scaffold, replace its guarded placeholder body with
  your implementation before choosing learner mode.

Change the setup flag before restarting. Record a boundary-case prediction
before each check and change an input after it passes. Later experiment cells
without a scaffold are reference demonstrations, not automated learner grades.

**Teaching companion:** [Worked examples, assumptions and self-checks](teaching_guide.md).
For additional independently graded tasks, see the
[progressive coding projects](../docs/INTERACTIVE_LEARNING.md).


**A bounded first investigation:** Make the grouping axis and verifier visible. Start with **Exercises 1, 2, 3 and 5** and the
[guide’s prediction, visual diagnostic and changed-input check](teaching_guide.md#notebook-investigation-make-the-grouping-axis-and-verifier-visible).
Save your own prediction before running the reference cells. For your own implementation, choose learner mode in setup, restart, and verify
that the corresponding check calls your completed function.


**Worked examples and derivations:** [Teaching guide](teaching_guide.md) — prerequisite links, analogy boundaries, calculated examples, code connections, and retrieval questions.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))
sys.path.insert(0, os.path.abspath('.'))
from common.plotting import use_rl_style, RL_PALETTE
from common.seed import seed_everything
use_rl_style(); seed_everything(0)

import numpy as np
import matplotlib.pyplot as plt
import torch

from project import solutions as S
from project.env import exact_match_reward, extract_final_number, ArithmeticTask
from project.train import grpo_advantages, rloo_advantages, train
# True runs documented references. False checks YOUR completed functions.
# Restart and run all after changing this flag; an unfinished learner stops.
REFERENCE_MODE = True
def select_exercise(reference, learner):
    return reference if REFERENCE_MODE else learner
print("Exercise mode:", "reference demonstration" if REFERENCE_MODE else "learner checks (no fallback)")


## Exercise 1 ($\bigstar$). Extract the final integer from a completion.

Write a function that mirrors the GSM8K answer-extraction convention:
given an arbitrary text completion, return the last integer (positive
or negative) appearing in the text, or `None` if no integer is present.

> **Hint** (try yourself first): Python's `re.findall(r'-?\d+', text)` returns every signed-integer match in order; the last element is what you want.

In [ ]:
def exercise_1(text):
    # YOUR CODE HERE
    raise NotImplementedError

In [ ]:
# The selected mode determines whether this check calls your implementation.
ref = select_exercise(S.exercise_1, exercise_1)
for t, expected in [("the answer is 17", 17), ("3 then 4 so 7", 7), ("nope", None), ("-2 + 3 = 1", 1)]:
    assert ref(t) == expected
print('exercise 1 ok')

## Exercise 2 ($\bigstar\bigstar$). RLOO advantage from a vector of group rewards.

Given a length-$G$ vector of rewards, return the length-$G$ vector of
RLOO advantages
$$\hat A_i = r_i - \frac{1}{G-1}\sum_{j \neq i} r_j = \frac{G}{G-1}(r_i - \bar r).$$

> **Hint**: $\sum_{j \neq i} r_j = (\sum_j r_j) - r_i$; the closed-form $G/(G-1)\,(r_i - \bar r)$ is the cleanest.

In [ ]:
def exercise_2(rewards):
    # YOUR CODE HERE
    raise NotImplementedError

In [ ]:
ref = select_exercise(S.exercise_2, exercise_2)
a = ref(np.array([1.0, 0.0, 1.0, 0.0]))
assert abs(a.sum()) < 1e-9, "RLOO advantages should sum to zero on symmetric input"
assert np.allclose(ref([1.0, 1.0, 1.0]), [0.0, 0.0, 0.0]), "constant rewards -> zero advantage"
print('exercise 2 ok')

## Exercise 3 ($\bigstar\bigstar$). GRPO advantage --- standard and Dr.GRPO variants.

Given a batch of group rewards $r \in \mathbb{R}^{B \times G}$, return
$(r_i - \bar r) / (\sigma_r + \varepsilon)$ (standard) or just
$(r_i - \bar r)$ (Dr.GRPO debiased).

In [ ]:
def exercise_3(rewards, debias=False, eps=1e-6):
    # YOUR CODE HERE
    raise NotImplementedError

In [ ]:
ref = select_exercise(S.exercise_3, exercise_3)
r = np.array([[0.0, 1.0, 0.0, 1.0]])
a = ref(r)
assert abs(a.mean()) < 1e-6
assert abs(a.std() - 1.0) < 1e-2
ad = ref(r, debias=True)
assert abs(ad.mean()) < 1e-12
print('exercise 3 ok')

## Exercise 4 ($\bigstar\bigstar$). Coefficient variance diagnostic.


Sample groups of Bernoulli rewards and compare raw-reward variance with within-group centred variance. This measures coefficients, not full policy-gradient variance. The self-including mean is action dependent; RLOO uses the other samples and multiplies centred coefficients by G/(G-1).
Sample $B$ groups of $G$ Bernoulli$(p)$ rewards and confirm
$\Var[r_i - \bar r] \leq \Var[r_i]$ --- the variance reduction lemma in
the notes.

In [ ]:
def exercise_4(rewards):
    # YOUR CODE HERE
    raise NotImplementedError

In [ ]:
ref = select_exercise(S.exercise_4, exercise_4)
rng = np.random.default_rng(0)
r = (rng.uniform(size=(500, 8)) < 0.3).astype(float)
v_r, v_c = ref(r)
assert v_c <= v_r + 1e-9, f"baseline failed: {v_c} > {v_r}"
print(f'exercise 4 ok --- v_reinforce={v_r:.4f}  v_centered={v_c:.4f}')
# Raw coefficients (0,0,1,1) have variance .25; centering each constant group gives0.
fixture_raw,fixture_centered=ref(np.array([[0.,0.],[1.,1.]]))
assert np.isclose(fixture_raw,.25) and np.isclose(fixture_centered,0.)


## Exercise 5 ($\bigstar\bigstar$). KL k3 estimator.

Schulman's low-variance unbiased KL estimator: given paired
log-probabilities $\log p$ and $\log q$ for samples drawn from $p$,
$k_3 = \exp(\log q - \log p) - (\log q - \log p) - 1$. The sample
mean of $k_3$ is unbiased for $\KL(p \,\|\, q)$ and always non-negative.

In [ ]:
def exercise_5(logp, logq):
    # YOUR CODE HERE
    raise NotImplementedError

In [ ]:
ref = select_exercise(S.exercise_5, exercise_5)
rng = np.random.default_rng(1)
logp = -rng.uniform(0.1, 1.0, size=200)
assert abs(ref(logp, logp)) < 1e-9
assert ref(logp, logp - 0.5) > 0
print('exercise 5 ok')

## Exercise 6 ($\bigstar\bigstar$). Token-level PPO clipped surrogate.

Return the (negative) clipped surrogate elementwise.

In [ ]:
def exercise_6(ratio, adv, clip_eps=0.2):
    # YOUR CODE HERE
    raise NotImplementedError

In [ ]:
ref = select_exercise(S.exercise_6, exercise_6)
loss = ref(np.array([2.0, 0.1, 1.0]), np.array([1.0, 1.0, 1.0]), 0.2)
assert abs(loss[0] - (-1.2)) < 1e-9
print('exercise 6 ok')

## Exercise 7 ($\bigstar\bigstar$). Process-reward aggregation (matching application).

Given a batch of step-reward sequences (think: per-interview-stage
scores in an ATS pipeline), aggregate each row into a single
trajectory score. Implement `min`, `mean`, and `prod` aggregators.
Which is appropriate for the ATS analogue, and why?

In [ ]:
def exercise_7(step_rewards, mode='min'):
    # YOUR CODE HERE
    raise NotImplementedError

In [ ]:
ref = select_exercise(S.exercise_7, exercise_7)
s = np.array([[0.9, 0.9, 0.2], [0.8, 0.8, 0.8]])
assert np.allclose(ref(s, 'min'), [0.2, 0.8])
assert np.allclose(ref(s, 'mean'), [0.6666667, 0.8])
assert np.allclose(ref(s, 'prod'), [0.162, 0.512])
print('exercise 7 ok')

## Exercise 8 ($\bigstar\bigstar\bigstar$, open-ended). Reward shaping and format hacking.

Implement a shaped reward that gives partial credit ($+\alpha$) for
emitting *any* parseable integer, plus the binary exact-match. Then
compare the policy's behaviour under the shaped and unshaped reward.
What do you observe? (Hint: count the rate at which the policy emits
*any* number vs the rate at which it emits the *correct* number.)

In [ ]:
def exercise_8(completion, target, fmt_bonus=0.1):
    # YOUR CODE HERE
    raise NotImplementedError

In [ ]:
ref = select_exercise(S.exercise_8, exercise_8)
assert ref('3', 3) >= 1.0  # correct gets base + bonus
assert ref('hello', 3) == 0.0  # nothing parseable, no reward
assert 0 < ref('answer 4', 3) < 1.0  # wrong number but parseable
print('exercise 8 ok (open-ended discussion in markdown below)')

## Visualisation --- training curve and variance comparison


The prompts come from the same finite task set used for training; this tests new samples, not generalisation to disjoint prompts.
Run the reference GRPO loop for a short pass and plot fresh-sample accuracy
and the variance-vs-group-size diagnostic. These are the same figures
used in `notes.pdf`.

In [ ]:
result = train(seed=0, num_steps=200, eval_every=50, batch_prompts=4, group_size=8, sft_warmup_steps=400)
h = result['history']
fig, ax = plt.subplots()
ax.plot(h['eval_step'], h['eval_acc'], marker='o', color=RL_PALETTE['blue'])
ax.set_xlabel('step'); ax.set_ylabel('held-out greedy accuracy')
ax.set_title('GRPO accuracy on toy arithmetic')
fig.tight_layout(); plt.show()
print(f"final accuracy: {result['final_acc']:.3f}")

In [ ]:
rng = np.random.default_rng(0)
Gs = [2, 4, 8, 16, 32]
vs = []
for G in Gs:
    r = (rng.uniform(size=(2000, G)) < 0.3).astype(float)
    _, v_c = select_exercise(S.exercise_4, exercise_4)(r)
    vs.append(v_c)
fig, ax = plt.subplots()
ax.plot(Gs, vs, marker='o', color=RL_PALETTE['teal'])
ax.set_xscale('log', base=2); ax.set_xlabel('group size G'); ax.set_ylabel('Var[r_i - mean(r)]')
ax.set_title('Centered estimator variance vs G (Bernoulli p=0.3)')
fig.tight_layout(); plt.show()

## Discussion (write your own)

1. How does the GRPO fresh-sample accuracy curve change if you set `sft_warmup_steps=0`? Explain in terms of the cold-start pathology of RLVR.
2. Try `variant='dr_grpo'` in the trainer config. Do you see shorter or longer completions on average? Why might that be?
3. Replace the verifier with `sympy_equiv_reward` from `project/env.py`. For this toy task it changes nothing --- why? In what task would it matter?

<!-- AUTO-HINTS-SECTION -->
## Hints toward answers

Attempt the exercise first. Before opening a hint, write the quantity you are
estimating, the data source, and the expected update direction. The
[teaching guide](teaching_guide.md) supplies worked examples, assumptions,
and transfer questions. The hints below collect the exercise's own inline
hints; they do not reveal or execute reference solutions.

<details>
<summary>Exercise 1 ($\bigstar$). Extract the final integer from a completion.</summary>

> **Hint** (try yourself first): Python's `re.findall(r'-?\d+', text)` returns every signed-integer match in order; the last element is what you want.

</details>

<details>
<summary>Exercise 2 ($\bigstar\bigstar$). RLOO advantage from a vector of group rewards.</summary>

> **Hint**: $\sum_{j \neq i} r_j = (\sum_j r_j) - r_i$; the closed-form $G/(G-1)\,(r_i - \bar r)$ is the cleanest.

</details>

<details>
<summary>Exercise 3 ($\bigstar\bigstar$). GRPO advantage --- standard and Dr.GRPO variants.</summary>

No inline hint is supplied for this exercise. Use the corresponding derivation and code bridge in the teaching guide.

</details>

<details>
<summary>Exercise 4 ($\bigstar\bigstar$). Coefficient variance diagnostic.</summary>

No inline hint is supplied for this exercise. Use the corresponding derivation and code bridge in the teaching guide.

</details>

<details>
<summary>Exercise 5 ($\bigstar\bigstar$). KL k3 estimator.</summary>

No inline hint is supplied for this exercise. Use the corresponding derivation and code bridge in the teaching guide.

</details>

<details>
<summary>Exercise 6 ($\bigstar\bigstar$). Token-level PPO clipped surrogate.</summary>

No inline hint is supplied for this exercise. Use the corresponding derivation and code bridge in the teaching guide.

</details>

<details>
<summary>Exercise 7 ($\bigstar\bigstar$). Process-reward aggregation (matching application).</summary>

No inline hint is supplied for this exercise. Use the corresponding derivation and code bridge in the teaching guide.

</details>

<details>
<summary>Exercise 8 ($\bigstar\bigstar\bigstar$, open-ended). Reward shaping and format hacking.</summary>

(Hint: count the rate at which the policy emits
*any* number vs the rate at which it emits the *correct* number.)

</details>

After solving, change one input or assumption and predict the result before
running it. In reference mode, passing checks verifies the demonstration. In learner mode,
the selected checks must pass against your completed functions without fallback.


## Statistical investigation — A group baseline can couple observations

**Goal.** Derive the self-inclusion bias of a same-group reward baseline. This bounded reference investigation runs on CPU without downloads.
Write your prediction in a new Markdown cell before executing it.

**Predict.** For independent Bernoulli completions with p=0.3 and score A-p, compare a
baseline using the same group's mean reward with a leave-one-out baseline.
Derive the expected gradient shrinkage (G-1)/G for the same-group baseline.
The independent unit for uncertainty is the entire prompt group.

**Work a small case by hand.** For p=0.3, the exact Bernoulli-logit policy gradient is
p(1-p)=0.21. A same-group mean contains the current reward with coefficient
1/G. Therefore E[(A-p)(A-mean reward)]=(1-1/G)×0.21: at G=2,4,8,16
the values are 0.105,0.1575,0.18375,0.196875. A leave-one-out baseline
excludes the current action and is independent of its score in this fixture,
so its expected gradient remains 0.21.

**Reading bridge:** [Probability and statistics lesson](../docs/statistics/05-resampling-dependence.md).
These are original course examples informed by the textbook sections in the
[statistics source catalog](../docs/statistics/SOURCES.md); textbook PDFs are not required to run them.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from common.plotting import RL_PALETTE, use_rl_style
use_rl_style()
rng_stats=np.random.default_rng(120);p=.3;groups=[2,4,8,16]
same_means=[];loo_means=[];target=p*(1-p)
for group_size in groups:
    a=rng_stats.binomial(1,p,(3000,group_size)).astype(float)
    same=a-a.mean(1,keepdims=True)
    loo=a-(a.sum(1,keepdims=True)-a)/(group_size-1)
    g_same=((a-p)*same).mean(1);g_loo=((a-p)*loo).mean(1)
    same_means.append(g_same.mean());loo_means.append(g_loo.mean())
    assert np.allclose(loo,same*group_size/(group_size-1))
    print('G=',group_size,'same-group gradient=',g_same.mean(),'LOO=',g_loo.mean(),
          'same-group exact=',target*(group_size-1)/group_size)
fig,ax=plt.subplots(figsize=(7,3.5))
ax.plot(groups,same_means,'o-',label='Same-group mean baseline')
ax.plot(groups,loo_means,'s-',label='Leave-one-out baseline')
ax.axhline(target,color=RL_PALETTE['gray'],ls='--',label='Exact policy gradient')
ax.set(xlabel='Completions per independent prompt group G',ylabel='Mean gradient over 3000 groups')
ax.legend();fig.tight_layout();plt.show()


**Interpret and transfer.** LOO removes this self-inclusion bias under the independent-completion
fixture. Group standard-deviation normalization changes the estimator again;
the calculation does not establish unbiasedness of an entire GRPO update.

**Transfer.** Make all completions in a group share a latent success rate.
Recalculate the relevant conditional expectation and explain why G rewards
from one prompt are not G independent task observations.


**Read your result.** Compare the printed averages with these exact expectations. Centered
coefficients can sum to zero while their score-weighted gradient does not.
The plot evaluates group-level gradient samples, not just reward-coefficient variance.

<details><summary>Worked feedback for the changed assumption</summary>

With a prompt-specific success probability p_j, condition first on that
prompt and use the correct score A-p_j. Conditional independence of its
completions gives the same identities with p_j(1-p_j); averaging across
prompts introduces task variation. Using A-0.3 for an unmodeled latent mixture
is a different score model and does not inherit this derivation automatically.

</details>
